In [1]:
pip install nibabel numpy pandas matplotlib scikit-learn scikit-image scipy 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#Essential Package imports:
import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import ndimage
from skimage.measure import label, regionprops
from pathlib import Path
import json
from  matplotlib.lines import Line2D


In [3]:
DATA_DIR = Path(r"D:\\Task01_BrainTumour")
 
# Case to analyse when RUN_ALL_CASES is False (e.g. BRATS_001).
CASE_ID = "BRATS_001"
 
# False = analyse only CASE_ID and show the figure.
# True  = analyse every case in imagesTr and save one CSV row per case.
RUN_ALL_CASES = True
 
# Where results (CSV, figures) are saved.
OUTPUT_DIR = Path("outputs")
 
IMAGES_DIR = DATA_DIR / "imagesTr"
LABELS_DIR = DATA_DIR / "labelsTr"
OUTPUT_DIR.mkdir(exist_ok=True)

In [4]:
# ------------------------------------------------------------
# 2. MSD LABEL DEFINITIONS
# ------------------------------------------------------------
 
# IMPORTANT: Task01_BrainTumour does NOT use the original BraTS
# label numbers (1, 2, 4). Its labels are:
#   0 = Background
#   1 = Peritumoural oedema
#   2 = Non-enhancing tumour (includes necrosis)
#   3 = Enhancing tumour
LABEL_EDEMA = 1
LABEL_NON_ENHANCING = 2
LABEL_ENHANCING = 3

In [5]:
# ------------------------------------------------------------
# 3. READ THE MODALITY ORDER FROM dataset.json
# ------------------------------------------------------------

def get_modality_indices(dataset_dir):
    """
    Read dataset.json and return a dictionary that maps each
    modality (FLAIR, T1, T1ce, T2) to its position (channel) in
    the 4th dimension of the image.
 
    Reading it from the file avoids hard-coding the channel order.
    For this dataset it is: 0=FLAIR, 1=T1w, 2=T1gd, 3=T2w.
    """
    json_path = dataset_dir / "dataset.json"
 
    if not json_path.exists():
        raise FileNotFoundError(f"Cannot find {json_path}")
 
    with open(json_path, "r") as f:
        modality_names = json.load(f)["modality"]
 
    # dataset.json names -> the short names used in this script
    name_map = {"flair": "FLAIR", "t1w": "T1", "t1gd": "T1ce", "t2w": "T2"}
 
    indices = {}
    for channel, raw_name in modality_names.items():
        short_name = name_map.get(raw_name.lower())
        if short_name is None:
            raise ValueError(f"Unexpected modality name: {raw_name}")
        indices[short_name] = int(channel)
 
    return indices

In [6]:
 # ------------------------------------------------------------
# 4. LOAD ONE CASE
# ------------------------------------------------------------
 
def load_nifti(file_path):
    """
    Load a NIfTI file and return its image object (header and
    affine) together with the numerical volume (float32).
    """
    if not file_path.exists():
        raise FileNotFoundError(f"Cannot find file: {file_path}")
 
    image = nib.load(str(file_path))
    volume = image.get_fdata(dtype=np.float32)
 
    return image, volume
 
 
def load_case(case_id, modality_indices):
    """
    Load the 4D image and the label mask of one case.
 
    Returns:
        image_obj : nibabel image (used for the affine matrix)
        modalities: dict {"T1": 3D array, "T1ce": ..., "T2": ..., "FLAIR": ...}
        seg       : 3D label array
        seg_obj   : nibabel image of the label mask
    """
    image_obj, image_4d = load_nifti(IMAGES_DIR / f"{case_id}.nii.gz")
    seg_obj, seg = load_nifti(LABELS_DIR / f"{case_id}.nii.gz")
 
    # The image must be 4D: (X, Y, Z, 4 modalities)
    if image_4d.ndim != 4:
        raise ValueError(
            f"{case_id}: expected a 4D image, got shape {image_4d.shape}"
        )

    # Split the 4D array into one 3D volume per modality
    modalities = {
        name: image_4d[..., idx] for name, idx in modality_indices.items()
    }
 
    # Check that the mask lines up with the image
    # (same grid size and same spatial position)
    if image_4d.shape[:3] != seg.shape:
        raise ValueError(
            f"{case_id}: image grid {image_4d.shape[:3]} does not match "
            f"label grid {seg.shape}"
        )
 
    if not np.allclose(image_obj.affine, seg_obj.affine, atol=1e-4):
        raise ValueError(f"{case_id}: image and label are not spatially aligned")
 
    return image_obj, modalities, seg, seg_obj

In [7]:
# ------------------------------------------------------------
# 5. CREATE THE THREE TUMOUR REGIONS
# ------------------------------------------------------------
 
def make_tumour_regions(seg):
    """
    Build the standard BraTS binary regions from the MSD labels.
 
    ET (enhancing tumour) : label 3
    TC (tumour core)      : labels 2 and 3
    WT (whole tumour)     : labels 1, 2 and 3
    """
    ET = seg == LABEL_ENHANCING
    TC = np.isin(seg, [LABEL_NON_ENHANCING, LABEL_ENHANCING])
    WT = np.isin(seg, [LABEL_EDEMA, LABEL_NON_ENHANCING, LABEL_ENHANCING])
    return ET, TC, WT

In [8]:
# ------------------------------------------------------------
# 6. VOLUME CALCULATION
# ------------------------------------------------------------
 
def calculate_volume(mask, voxel_volume_mm3):
    """
    Return (voxel count, volume in mm3, volume in cm3) of a binary mask.
    """
    voxel_count = int(np.count_nonzero(mask))
    volume_mm3 = voxel_count * voxel_volume_mm3
    volume_cm3 = volume_mm3 / 1000.0
    return voxel_count, volume_mm3, volume_cm3

In [9]:
# ------------------------------------------------------------
# 7. INTENSITY FEATURES
# ------------------------------------------------------------
 
def extract_intensity_features(image, mask, modality):
    """
    Mean, median, standard deviation, 10th and 90th percentile of the
    MRI intensities inside a mask, for one modality.
    """
    values = image[mask]
 
    if values.size == 0:
        return {
            f"{modality}_mean": np.nan,
            f"{modality}_median": np.nan,
            f"{modality}_std": np.nan,
            f"{modality}_p10": np.nan,
            f"{modality}_p90": np.nan,
        }
 
    return {
        f"{modality}_mean": float(np.mean(values)),
        f"{modality}_median": float(np.median(values)),
        f"{modality}_std": float(np.std(values)),
        f"{modality}_p10": float(np.percentile(values, 10)),
        f"{modality}_p90": float(np.percentile(values, 90)),
    }

In [10]:
 # ------------------------------------------------------------
# 8. FEATURE EXTRACTION FOR ONE CASE
# ------------------------------------------------------------
 
def extract_features(case_id, modality_indices):
    """
    Run the full feature extraction for one case.
 
    Returns:
        results   : dictionary of all extracted features (one CSV row)
        modalities: the 3D MRI volumes (needed for plotting)
        masks     : (ET, TC, WT) binary masks (needed for plotting)
    """
    image_obj, modalities, seg, _ = load_case(case_id, modality_indices)
    ET, TC, WT = make_tumour_regions(seg)
 
    # --- Voxel volume -------------------------------------------------
    # The determinant of the 3x3 part of the affine matrix gives the
    # physical size of one voxel in mm3. In this dataset it is ~1 mm3.
    voxel_volume = abs(np.linalg.det(image_obj.affine[:3, :3]))
 
    # --- Volumes -------------------------------------------------------
    ET_count, ET_mm3, ET_cm3 = calculate_volume(ET, voxel_volume)
    TC_count, TC_mm3, TC_cm3 = calculate_volume(TC, voxel_volume)
    WT_count, WT_mm3, WT_cm3 = calculate_volume(WT, voxel_volume)

    
    # --- Tumour composition (fraction of the whole tumour) ------------
    if WT_count > 0:
        enhancing_fraction = ET_count / WT_count
        core_fraction = TC_count / WT_count
        edema_fraction = np.count_nonzero(seg == LABEL_EDEMA) / WT_count
    else:
        enhancing_fraction = core_fraction = edema_fraction = np.nan
 
    # --- Centroid and bounding box of the whole tumour ----------------
    coordinates = np.argwhere(WT)  # (N, 3) voxel indices of tumour voxels
 
    if len(coordinates) > 0:
        centroid_voxel = coordinates.mean(axis=0)
        # Convert voxel indices into physical (world) coordinates in mm
        centroid_world = nib.affines.apply_affine(
            image_obj.affine, centroid_voxel
        )
        bounding_dimensions = (
            coordinates.max(axis=0) - coordinates.min(axis=0) + 1
        )
    else:
        centroid_voxel = np.full(3, np.nan)
        centroid_world = np.full(3, np.nan)
        bounding_dimensions = np.full(3, np.nan)
 
    # --- Intensity statistics inside the whole tumour -----------------
    intensity_features = {}
    for modality_name, modality_image in modalities.items():
        intensity_features.update(
            extract_intensity_features(modality_image, WT, modality_name)
        )

    # --- Connected components (shape feature) -------------------------
    # connectivity=3 treats voxels touching by a face, edge or corner
    # as connected. Several separate lumps = a multifocal tumour.
    components = label(WT.astype(np.uint8), connectivity=3)
    properties = regionprops(components)
 
    if properties:
        largest_component_voxels = max(r.area for r in properties)
        number_of_components = len(properties)
    else:
        largest_component_voxels = 0
        number_of_components = 0
 
    # --- Compile everything into one dictionary -----------------------
    results = {
        "Case_ID": case_id,
        "Voxel_volume_mm3": voxel_volume,
 
        "ET_voxels": ET_count,
        "ET_volume_mm3": ET_mm3,
        "ET_volume_cm3": ET_cm3,
 
        "TC_voxels": TC_count,
        "TC_volume_mm3": TC_mm3,
        "TC_volume_cm3": TC_cm3,
 
        "WT_voxels": WT_count,
        "WT_volume_mm3": WT_mm3,
        "WT_volume_cm3": WT_cm3,
 
        "Enhancing_fraction": enhancing_fraction,
        "Tumour_core_fraction": core_fraction,
        "Oedema_fraction": edema_fraction,

        "Centroid_voxel_X": centroid_voxel[0],
        "Centroid_voxel_Y": centroid_voxel[1],
        "Centroid_voxel_Z": centroid_voxel[2],
 
        "Centroid_world_X_mm": centroid_world[0],
        "Centroid_world_Y_mm": centroid_world[1],
        "Centroid_world_Z_mm": centroid_world[2],
 
        "Bounding_box_X_voxels": bounding_dimensions[0],
        "Bounding_box_Y_voxels": bounding_dimensions[1],
        "Bounding_box_Z_voxels": bounding_dimensions[2],
 
        "Number_of_components": number_of_components,
        "Largest_component_voxels": largest_component_voxels,
    }
    results.update(intensity_features)
 
    return results, modalities, (ET, TC, WT)


In [11]:
 # ------------------------------------------------------------
# 9. VISUALISATION
# ------------------------------------------------------------
 
def visualise_case(case_id, modalities, masks, show=True):
    """
    Plot the axial slice with the most tumour on T1ce, with and
    without the ET / TC / WT outlines, and save it as a PNG.
    """
    ET, TC, WT = masks
 
    # Pick the axial slice containing the most whole-tumour voxels
    slice_counts = np.sum(WT, axis=(0, 1))
    best_slice = int(np.argmax(slice_counts))
 
    mri_slice = modalities["T1ce"][:, :, best_slice]
 
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
 
    # Left: the plain MRI slice
    axes[0].imshow(mri_slice.T, cmap="gray", origin="lower")
    axes[0].set_title(f"{case_id}: T1ce, axial slice {best_slice}")
    axes[0].axis("off")
 
    # Right: the same slice with tumour outlines
    axes[1].imshow(mri_slice.T, cmap="gray", origin="lower")
    
    
    # contour() needs a region to exist on this slice; skip empty ones
    for mask, colour in ((WT, "yellow"), (TC, "cyan"), (ET, "red")):
        mask_slice = mask[:, :, best_slice]
        if mask_slice.any():
            axes[1].contour(
                mask_slice.T, levels=[0.5], colors=[colour], origin="lower"
            )
 
    axes[1].set_title("Tumour segmentation: WT, TC and ET")
    axes[1].axis("off")
 
    axes[1].legend(
        handles=[
            Line2D([0], [0], color="yellow", label="Whole tumour (WT)"),
            Line2D([0], [0], color="cyan", label="Tumour core (TC)"),
            Line2D([0], [0], color="red", label="Enhancing tumour (ET)"),
        ],
        loc="lower right",
        fontsize=8,
    )
 
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / f"{case_id}_segmentation.png",
        dpi=300,
        bbox_inches="tight",
    )
 
    if show:
        plt.show()
    plt.close(fig)

In [12]:
# ------------------------------------------------------------
# 10. MAIN PROGRAM
# ------------------------------------------------------------
 
def main():
    modality_indices = get_modality_indices(DATA_DIR)
    print("Modality channels:", modality_indices)
 
    if RUN_ALL_CASES:
        # Every image in imagesTr. The "BRATS_*" pattern skips hidden
        # files such as "._BRATS_001.nii.gz" created by macOS.
        case_ids = sorted(
            p.name.replace(".nii.gz", "")
            for p in IMAGES_DIR.glob("BRATS_*.nii.gz")
        )
        print(f"Found {len(case_ids)} cases.")
    else:
        case_ids = [CASE_ID]
 
    all_results = []
 
    for i, case_id in enumerate(case_ids, start=1):
        print(f"[{i}/{len(case_ids)}] Processing {case_id}...")
 
        try:
            results, modalities, masks = extract_features(
                case_id, modality_indices
            )
        except Exception as error:
            # One bad file should not stop a long batch run
            print(f"  Skipped {case_id}: {error}")
            continue
 
        all_results.append(results)
 
        # Single-case mode: show the figure. Batch mode: save only.
        visualise_case(case_id, modalities, masks, show=not RUN_ALL_CASES)
 
    if not all_results:
        print("No cases were processed successfully.")
        return
 
    feature_table = pd.DataFrame(all_results)
 
    if not RUN_ALL_CASES:
        print("\nBRAIN TUMOUR FEATURE EXTRACTION RESULTS")
        print(feature_table.T)
 
    csv_name = "brain_tumour_features_all.csv" if RUN_ALL_CASES \
        else f"{CASE_ID}_features.csv"
    feature_table.to_csv(OUTPUT_DIR / csv_name, index=False)
 
    print(f"\nFeature extraction completed. Results saved to "
          f"{OUTPUT_DIR / csv_name}")
 
 
if __name__ == "__main__":
    main()

Modality channels: {'FLAIR': 0, 'T1': 1, 'T1ce': 2, 'T2': 3}
Found 484 cases.
[1/484] Processing BRATS_001...
[2/484] Processing BRATS_002...
[3/484] Processing BRATS_003...
[4/484] Processing BRATS_004...
[5/484] Processing BRATS_005...
[6/484] Processing BRATS_006...
[7/484] Processing BRATS_007...
[8/484] Processing BRATS_008...
[9/484] Processing BRATS_009...
[10/484] Processing BRATS_010...
[11/484] Processing BRATS_011...
[12/484] Processing BRATS_012...
[13/484] Processing BRATS_013...
[14/484] Processing BRATS_014...
[15/484] Processing BRATS_015...
[16/484] Processing BRATS_016...
[17/484] Processing BRATS_017...
[18/484] Processing BRATS_018...
[19/484] Processing BRATS_019...
[20/484] Processing BRATS_020...
[21/484] Processing BRATS_021...
[22/484] Processing BRATS_022...
[23/484] Processing BRATS_023...
[24/484] Processing BRATS_024...
[25/484] Processing BRATS_025...
[26/484] Processing BRATS_026...
[27/484] Processing BRATS_027...
[28/484] Processing BRATS_028...
[29/484